# Notebook 01 — ODE Biological Models

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 1 of 12  
**Time estimate:** 75 minutes

> Most biological dynamics are captured first by ordinary differential equations
> (ODEs). Population ecology, epidemic spread, gene circuit dynamics, and enzyme
> kinetics all start here. This notebook builds the ODE solver from scratch,
> then applies it to the three most important biological ODE systems.

---
## Step 1 — Motivation

The SIR epidemic model predicted herd immunity thresholds for COVID-19.
Lotka-Volterra equations predicted the oscillation of lynx and hare populations
that was observed in Hudson Bay Company fur records from 1845–1935. ODE-based
models are the first language of biological dynamics — not because they are perfect
(they assume large populations and no randomness) but because they are analytically
tractable and produce falsifiable predictions.

---
## Step 2 — Intuition

An ODE says: the *rate of change* of a quantity depends on its *current state*.
In population biology:
- dN/dt = rN (exponential growth: the more rabbits, the faster rabbits multiply)
- dN/dt = rN(1 - N/K) (logistic: growth slows as population approaches carrying capacity K)

**Numerical integration:** we can't always solve ODEs analytically. Instead, we
approximate the solution by stepping forward in small time increments Δt:
- **Euler method:** $y(t+\Delta t) \approx y(t) + f(t, y) \cdot \Delta t$  
  Simple but accumulates error. Unstable for stiff systems.
- **Runge-Kutta 4 (RK4):** uses four slope estimates per step to achieve 4th-order
  accuracy. The workhorse of numerical ODE integration.

**Phase portraits:** plot the state space (e.g., prey vs. predator) rather than time.
Fixed points, limit cycles, and stability are visible without solving for t.

---
## Step 3 — Biological Background

**Lotka-Volterra (1920s):** Alfred Lotka and Vito Volterra independently derived
the same ODE system for predator-prey dynamics. The equations predict oscillating
populations, confirmed qualitatively in lynx-hare data. Limitations: assumes prey
grow without bound in the absence of predators (no carrying capacity), and predators
eat with no satiation (no Holling functional response).

**SIR epidemic model (Kermack-McKendrick 1927):** divides the population into
Susceptible, Infectious, Recovered compartments. The key insight: not everyone
needs to be immune to stop an epidemic — once enough people recover (or are
vaccinated), the effective reproduction number $R_t = R_0 \cdot S/N$ drops below 1.
This is the herd immunity threshold: $1 - 1/R_0$ fraction must be immune.

**Nullclines and fixed points:** a nullcline is the set of states where
$\dot{x} = 0$ or $\dot{y} = 0$. Where two nullclines cross: a fixed point.
The nature of the fixed point (stable/unstable, node/spiral/saddle) determines
the long-run behaviour of the system.

---
## Step 4 — Mathematical Explanation

**Lotka-Volterra:**
$$\frac{dV}{dt} = rV - aVP, \quad \frac{dP}{dt} = baVP - dP$$
$V$: prey (Victims), $P$: predators; $r$: prey growth rate, $a$: predation rate,
$b$: predator conversion efficiency, $d$: predator death rate.
Fixed points: $(0,0)$ (unstable) and $(d/ba,\, r/a)$ (centre — neutral stability).

**SIR model:**
$$\frac{dS}{dt} = -\beta \frac{SI}{N}, \quad \frac{dI}{dt} = \beta \frac{SI}{N} - \gamma I, \quad \frac{dR}{dt} = \gamma I$$
$N = S+I+R$ (constant). Basic reproduction number: $R_0 = \beta / \gamma$.
Epidemic occurs iff $R_0 \cdot S_0/N > 1$. Final epidemic size satisfies:
$r_\infty = 1 - e^{-R_0 r_\infty}$ (transcendental equation, solved numerically).

**RK4 step:**
$$k_1 = f(t, y), \quad k_2 = f(t+h/2,\, y+hk_1/2)$$
$$k_3 = f(t+h/2,\, y+hk_2/2), \quad k_4 = f(t+h,\, y+hk_3)$$
$$y(t+h) = y(t) + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$
Local truncation error $O(h^5)$; global error $O(h^4)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.random.seed(42)

# ---- Numerical solvers from scratch ----
def euler(f, y0, t_span, dt):
    """Euler method ODE solver."""
    t0, tf = t_span
    t = np.arange(t0, tf + dt, dt)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        y[i+1] = y[i] + dt * np.array(f(t[i], y[i]))
    return t, y

def rk4(f, y0, t_span, dt):
    """Runge-Kutta 4 ODE solver."""
    t0, tf = t_span
    t = np.arange(t0, tf + dt, dt)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        k1 = np.array(f(t[i],          y[i]))
        k2 = np.array(f(t[i] + dt/2,   y[i] + dt*k1/2))
        k3 = np.array(f(t[i] + dt/2,   y[i] + dt*k2/2))
        k4 = np.array(f(t[i] + dt,     y[i] + dt*k3))
        y[i+1] = y[i] + (dt/6) * (k1 + 2*k2 + 2*k3 + k4)
    return t, y

# ---- Lotka-Volterra ----
def lotka_volterra(t, y, r=1.0, a=0.1, b=0.1, d=1.5):
    V, P = y
    dV = r*V - a*V*P
    dP = b*a*V*P - d*P
    return [dV, dP]

t_lv, y_lv_euler = euler(lotka_volterra, [40, 9], (0, 30), dt=0.01)
_, y_lv_rk4   = rk4(  lotka_volterra, [40, 9], (0, 30), dt=0.01)

# SciPy reference
lv_ref = solve_ivp(lotka_volterra, (0, 30), [40, 9], max_step=0.01, dense_output=True)
t_ref = np.linspace(0, 30, 3001)
y_ref = lv_ref.sol(t_ref)

# ---- SIR model ----
def sir(t, y, beta=0.3, gamma=0.1, N=10000):
    S, I, R = y
    dS = -beta * S * I / N
    dI =  beta * S * I / N - gamma * I
    dR =  gamma * I
    return [dS, dI, dR]

N = 10000
_, y_sir = rk4(lambda t, y: sir(t, y, beta=0.3, gamma=0.1, N=N),
               [N-1, 1, 0], (0, 160), dt=0.1)

R0 = 0.3 / 0.1
herd_immunity = 1 - 1/R0
print(f'SIR model: R0={R0:.1f}')
print(f'Herd immunity threshold: {herd_immunity*100:.0f}% of population')
print(f'Peak infected: {y_sir[:, 1].max():.0f} ({y_sir[:, 1].max()/N*100:.1f}%)')
print(f'Final recovered fraction: {y_sir[-1, 2]/N:.3f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel A: LV time series — Euler vs RK4 vs SciPy
ax = axes[0, 0]
ax.plot(t_ref, y_ref[0], 'k-', lw=2, label='SciPy (reference)')
ax.plot(t_lv, y_lv_rk4[:, 0], 'steelblue', lw=1.5, ls='--', label='RK4 (scratch)')
ax.plot(t_lv, y_lv_euler[:, 0], 'tomato', lw=1, ls=':', label='Euler (scratch)')
ax.set_xlabel('Time'); ax.set_ylabel('Prey population')
ax.set_title('A. Lotka-Volterra prey\nEuler vs RK4 vs SciPy')
ax.legend(fontsize=8)

# Panel B: LV phase portrait
ax = axes[0, 1]
for y0, col in [([40, 9], 'steelblue'), ([20, 5], 'tomato'), ([60, 14], 'green')]:
    _, y_ph = rk4(lotka_volterra, y0, (0, 30), dt=0.01)
    ax.plot(y_ph[:, 0], y_ph[:, 1], color=col, lw=1.5)
    ax.plot(y_ph[0, 0], y_ph[0, 1], 'o', color=col, ms=6)
# Fixed point
ax.plot(1.5/0.1, 1.0/0.1, 'k*', ms=12, label='Fixed point (d/ab, r/a)')
# Nullclines
V_range = np.linspace(1, 80, 200)
P_V_null = np.full_like(V_range, 1.0/0.1)  # V-nullcline: P = r/a
V_P_null = np.full_like(V_range, 1.5/0.1)  # P-nullcline: V = d/(ba)
ax.axhline(10.0, color='steelblue', ls=':', lw=1, label='V-nullcline (dV/dt=0)')
ax.axvline(15.0, color='tomato', ls=':', lw=1, label='P-nullcline (dP/dt=0)')
ax.set_xlabel('Prey (V)'); ax.set_ylabel('Predator (P)')
ax.set_title('B. LV phase portrait\n(orbits + nullclines)')
ax.legend(fontsize=7)

# Panel C: RK4 error vs step size
dt_vals = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005]
errors = []
for dt in dt_vals:
    _, y_test = rk4(lotka_volterra, [40, 9], (0, 5), dt=dt)
    _, y_fine = rk4(lotka_volterra, [40, 9], (0, 5), dt=0.001)
    # Error at t=5
    n_steps_test = int(5/dt)
    n_steps_fine = int(5/0.001)
    errors.append(abs(y_test[min(n_steps_test, len(y_test)-1), 0] - y_fine[min(n_steps_fine, len(y_fine)-1), 0]))
ax = axes[0, 2]
ax.loglog(dt_vals, errors, 'o-', color='steelblue', lw=2)
# Reference O(h^4) line
dts = np.array(dt_vals)
ax.loglog(dts, errors[0] * (dts/dts[0])**4, 'k--', lw=1, label='O(h⁴)')
ax.set_xlabel('Step size h'); ax.set_ylabel('Error at t=5')
ax.set_title('C. RK4 convergence\n(4th-order accuracy)')
ax.legend(fontsize=9)

# Panel D: SIR epidemic curve
ax = axes[1, 0]
t_sir = np.arange(0, 160.1, 0.1)
ax.plot(t_sir, y_sir[:, 0]/N, 'steelblue', lw=2, label='S (susceptible)')
ax.plot(t_sir, y_sir[:, 1]/N, 'tomato', lw=2, label='I (infectious)')
ax.plot(t_sir, y_sir[:, 2]/N, 'green', lw=2, label='R (recovered)')
ax.axhline(1 - 1/R0, color='grey', ls='--', lw=1, label=f'Herd immunity ({(1-1/R0)*100:.0f}%)')
peak_t = t_sir[np.argmax(y_sir[:, 1])]
ax.axvline(peak_t, color='tomato', ls=':', lw=1)
ax.set_xlabel('Days'); ax.set_ylabel('Fraction of population')
ax.set_title(f'D. SIR epidemic (R₀={R0:.1f})\nβ=0.3, γ=0.1, N={N}')
ax.legend(fontsize=8)

# Panel E: R0 sensitivity — final epidemic size
R0_vals = np.linspace(1.01, 5.0, 200)
final_sizes = []
for r0 in R0_vals:
    beta = r0 * 0.1
    _, y = rk4(lambda t, y, b=beta: sir(t, y, beta=b, gamma=0.1, N=1000),
               [999, 1, 0], (0, 500), dt=0.5)
    final_sizes.append(y[-1, 2] / 1000)
ax = axes[1, 1]
ax.plot(R0_vals, final_sizes, 'tomato', lw=2)
ax.axhline(0, color='k', lw=0.5)
ax.fill_between(R0_vals, final_sizes, 0, alpha=0.2, color='tomato')
ax.set_xlabel('Basic reproduction number R₀')
ax.set_ylabel('Final epidemic size (fraction infected)')
ax.set_title('E. R₀ vs. final epidemic size\n(herd immunity: intercept at R₀=1)')

# Panel F: Multiple SIR scenarios
ax = axes[1, 2]
scenarios = [
    ('No intervention', 0.3, 0.1),
    ('50% contact reduction', 0.15, 0.1),
    ('Faster recovery (treatment)', 0.3, 0.2),
]
colors = ['tomato', 'steelblue', 'green']
for (name, beta, gamma), col in zip(scenarios, colors):
    _, y = rk4(lambda t, y, b=beta, g=gamma: sir(t, y, beta=b, gamma=g, N=N),
               [N-1, 1, 0], (0, 200), dt=0.1)
    ax.plot(np.arange(0, 200.1, 0.1), y[:, 1]/N, color=col, lw=2,
              label=f'{name} (R₀={beta/gamma:.1f})')
ax.set_xlabel('Days'); ax.set_ylabel('Fraction infectious')
ax.set_title('F. Intervention comparison\n(I(t) under different scenarios)')
ax.legend(fontsize=8)

plt.suptitle('Module 15 NB01: ODE Biological Models', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ode_biological_models.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement the SEIR model (add Exposed compartment between S and I, with
   incubation rate σ). How does the epidemic curve change compared to SIR?
2. Find the fixed points of the Lotka-Volterra system algebraically. Show
   the Jacobian at each fixed point and classify it (stable/unstable, node/spiral/saddle).
3. The Euler method is unstable for stiff ODEs. Demonstrate this: use Euler
   with dt=0.5 on the SIR model. What happens?

---
## Step 10 — Quiz

1. What is R₀ and how does it relate to β and γ in the SIR model?
2. What does a nullcline represent? How do you find the fixed points of a
   2D ODE system using nullclines?
3. Why does RK4 use four slope estimates rather than one (Euler)?
   What order of accuracy does each method achieve?
4. In the Lotka-Volterra model, the fixed point is a centre (neutral stability),
   not a stable spiral. What does this mean for the real biological system?

---
## Step 12 — Reflection

> *[In one paragraph: explain the herd immunity threshold using the SIR model.
> Start from the condition dI/dt < 0, derive the threshold mathematically,
> and explain what it means in plain English for a disease with R₀=3 (like measles).]*

---
*Next: `02_stochastic_simulation_gillespie.ipynb`*